In [ ]:
"""
# EXPLORATORY DATA ANALYSIS NOTEBOOK
# CSC 3221 - Introduction to Data Science
# ICT University Cameroon
# Mobile Money Transaction Analysis - User Classification
"""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

In [ ]:
os.makedirs('../3_EDA/visualizations', exist_ok=True)

In [ ]:
PALETTE = {'High': '#028090', 'Medium': '#F0A500', 'Low': '#E63946'}
COLORS = ['#028090', '#02C39A', '#F0A500', '#E63946', '#6B7280', '#8B5CF6', '#EC4899', '#14B8A6']
plt.rcParams.update({'font.family': 'DejaVu Sans', 'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.labelsize': 11, 'axes.titlesize': 13,
                     'xtick.labelsize': 9, 'ytick.labelsize': 9})

In [ ]:
final_df = pd.read_csv('../2_Data_Cleaning/cleaned_data.csv')
tx = pd.read_csv('../2_Data_Cleaning/cleaned_transactions.csv')
user_feat = pd.read_csv('../2_Data_Cleaning/user_features.csv')
tx['transaction_date'] = pd.to_datetime(tx['transaction_date'])

In [ ]:
print(f"Dataset loaded: {final_df.shape[0]} users, {len(tx)} transactions")
print(f"Segment distribution:\n{final_df['activity_segment'].value_counts()}")

In [ ]:
# ============================================================
# CHART 1: Distribution Plots - Transactions per Month & Avg Amount
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribution of Key Numerical Variables', fontsize=15, fontweight='bold', y=1.01)

In [ ]:
for ax, col, label, color in zip(
    axes,
    ['tx_per_month', 'avg_transaction_amount'],
    ['Transactions per Month', 'Average Transaction Amount (XAF)'],
    [COLORS[0], COLORS[2]]
):
    ax.hist(final_df[col].dropna(), bins=18, color=color, alpha=0.8, edgecolor='white', linewidth=0.5)
    mean_val = final_df[col].mean()
    median_val = final_df[col].median()
    ax.axvline(mean_val, color='#1E293B', linewidth=1.5, linestyle='--', label=f'Mean: {mean_val:.1f}')
    ax.axvline(median_val, color='#E63946', linewidth=1.5, linestyle=':', label=f'Median: {median_val:.1f}')
    ax.set_xlabel(label)
    ax.set_ylabel('Number of Users')
    ax.legend(fontsize=9)
    ax.set_facecolor('#F8FAFC')

In [ ]:
plt.tight_layout()
plt.savefig('visualizations/01_distribution_plots.png', bbox_inches='tight', dpi=150)
plt.close()
print("Chart 1 saved: distribution plots")

In [ ]:
# ============================================================
# CHART 2: Box Plots by Activity Segment
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Transaction Metrics by Activity Segment', fontsize=15, fontweight='bold')

In [ ]:
metrics = [
    ('total_transactions', 'Total Transactions'),
    ('avg_transaction_amount', 'Avg Transaction Amount (XAF)'),
    ('tx_per_month', 'Transactions per Month')
]
order = ['Low', 'Medium', 'High']
pal = [PALETTE[s] for s in order]

In [ ]:
for ax, (col, label) in zip(axes, metrics):
    data_by_seg = [final_df[final_df['activity_segment'] == s][col].dropna() for s in order]
    bp = ax.boxplot(data_by_seg, labels=order, patch_artist=True,
                    medianprops={'color': 'white', 'linewidth': 2},
                    whiskerprops={'linewidth': 1.2},
                    capprops={'linewidth': 1.2})
    for patch, color in zip(bp['boxes'], pal):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    ax.set_ylabel(label)
    ax.set_xlabel('Activity Segment')
    ax.set_facecolor('#F8FAFC')

In [ ]:
plt.tight_layout()
plt.savefig('visualizations/02_boxplots_segments.png', bbox_inches='tight', dpi=150)
plt.close()
print("Chart 2 saved: box plots")

In [ ]:
# ============================================================
# CHART 3: Time Series - Monthly Transaction Volume
# ============================================================
tx_success = tx[tx['status'] == 'Success'].copy()
tx_success['ym'] = tx_success['transaction_date'].dt.to_period('M')
monthly = tx_success.groupby('ym').agg(
    total_amount=('amount_xaf', 'sum'),
    tx_count=('transaction_id', 'count')
).reset_index()
monthly['ym_str'] = monthly['ym'].astype(str)

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

In [ ]:
bars = ax1.bar(range(len(monthly)), monthly['total_amount'] / 1e6,
               color=COLORS[0], alpha=0.7, label='Total Amount (M XAF)')
line = ax2.plot(range(len(monthly)), monthly['tx_count'],
                color=COLORS[2], linewidth=2.5, marker='o', markersize=5,
                label='Transaction Count')

In [ ]:
ax1.set_xticks(range(len(monthly)))
ax1.set_xticklabels(monthly['ym_str'], rotation=45, ha='right', fontsize=8)
ax1.set_ylabel('Total Amount (Millions XAF)', color=COLORS[0])
ax2.set_ylabel('Number of Transactions', color=COLORS[2])
ax1.tick_params(axis='y', labelcolor=COLORS[0])
ax2.tick_params(axis='y', labelcolor=COLORS[2])
ax1.set_title('Monthly Transaction Volume & Count (Mar 2024 - Feb 2025)', fontsize=14, fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
ax1.set_facecolor('#F8FAFC')
fig.tight_layout()
plt.savefig('visualizations/03_time_series.png', bbox_inches='tight', dpi=150)
plt.close()
print("Chart 3 saved: time series")

In [ ]:
# ============================================================
# CHART 4: Correlation Heatmap
# ============================================================
num_cols = ['age', 'household_size', 'years_using_mm', 'total_transactions',
            'avg_transaction_amount', 'tx_per_month', 'send_receive_ratio',
            'weekend_ratio', 'days_since_last_tx', 'tx_type_diversity',
            'total_fees_paid', 'activity_label']
corr_df = final_df[num_cols].dropna()
corr = corr_df.corr()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
cmap = sns.diverging_palette(220, 10, as_cmap=True)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap=cmap,
            vmin=-1, vmax=1, center=0, linewidths=0.5, linecolor='white',
            annot_kws={'size': 8}, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap of Numerical Features', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('visualizations/04_correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.close()
print("Chart 4 saved: correlation heatmap")

In [ ]:
# ============================================================
# CHART 5: Bar Charts - Profession, Zone, Transaction Types
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Categorical Variable Distributions', fontsize=15, fontweight='bold')

In [ ]:
# Profession by segment
prof_seg = final_df.groupby(['profession', 'activity_segment']).size().unstack(fill_value=0)
prof_seg = prof_seg.reindex(columns=['Low', 'Medium', 'High'])
prof_seg.plot(kind='bar', ax=axes[0], color=[PALETTE[s] for s in ['Low', 'Medium', 'High']],
              edgecolor='white', linewidth=0.5)
axes[0].set_title('Users by Profession & Segment')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(title='Segment', fontsize=8)
axes[0].set_facecolor('#F8FAFC')

In [ ]:
# Zone type distribution
zone_counts = final_df['zone_type'].value_counts()
axes[1].bar(zone_counts.index, zone_counts.values,
            color=[COLORS[0], COLORS[1], COLORS[2]], edgecolor='white')
axes[1].set_title('Users by Geographic Zone Type')
axes[1].set_ylabel('Number of Users')
axes[1].set_facecolor('#F8FAFC')
for bar in axes[1].patches:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(int(bar.get_height())), ha='center', fontsize=10, fontweight='bold')

In [ ]:
# Transaction types
tx_type_counts = tx_success['transaction_type'].value_counts()
colors_bar = COLORS[:len(tx_type_counts)]
axes[2].barh(tx_type_counts.index, tx_type_counts.values, color=colors_bar, edgecolor='white')
axes[2].set_title('Transaction Types (Successful)')
axes[2].set_xlabel('Count')
axes[2].set_facecolor('#F8FAFC')

In [ ]:
plt.tight_layout()
plt.savefig('visualizations/05_bar_charts.png', bbox_inches='tight', dpi=150)
plt.close()
print("Chart 5 saved: bar charts")

In [ ]:
# ============================================================
# CHART 6: Scatter Plot - Income vs Transaction Volume
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))
for seg, color in PALETTE.items():
    subset = final_df[final_df['activity_segment'] == seg]
    ax.scatter(subset['income_encoded'] + np.random.uniform(-0.15, 0.15, len(subset)),
               subset['avg_monthly_amount'],
               color=color, alpha=0.75, s=80, label=seg, edgecolors='white', linewidth=0.5)

In [ ]:
income_labels = ['<50K', '50-100K', '100-200K', '200-500K', '>500K']
ax.set_xticks(range(5))
ax.set_xticklabels(income_labels)
ax.set_xlabel('Monthly Income Range (XAF)')
ax.set_ylabel('Average Monthly Transaction Amount (XAF)')
ax.set_title('Income Level vs Average Monthly Transaction Amount', fontsize=14, fontweight='bold')
ax.legend(title='Activity Segment', fontsize=10)
ax.set_facecolor('#F8FAFC')
plt.tight_layout()
plt.savefig('visualizations/06_scatter_income_amount.png', bbox_inches='tight', dpi=150)
plt.close()
print("Chart 6 saved: scatter plot")

In [ ]:
# ============================================================
# CHART 7: Grouped Comparisons
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Transaction Behavior by Key Demographics', fontsize=15, fontweight='bold')

In [ ]:
# Tx per month by gender and segment
gender_seg = final_df.groupby(['gender', 'activity_segment'])['tx_per_month'].mean().unstack()
gender_seg.plot(kind='bar', ax=axes[0], color=[PALETTE[s] for s in ['Low', 'Medium', 'High']],
                edgecolor='white')
axes[0].set_title('Avg Transactions/Month by Gender & Segment')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Avg Transactions per Month')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Segment', fontsize=9)
axes[0].set_facecolor('#F8FAFC')

In [ ]:
# Avg amount by zone type and segment
zone_seg = final_df.groupby(['zone_type', 'activity_segment'])['avg_transaction_amount'].mean().unstack()
zone_seg.plot(kind='bar', ax=axes[1], color=[PALETTE[s] for s in ['Low', 'Medium', 'High']],
              edgecolor='white')
axes[1].set_title('Avg Transaction Amount by Zone Type & Segment')
axes[1].set_xlabel('Zone Type')
axes[1].set_ylabel('Avg Transaction Amount (XAF)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Segment', fontsize=9)
axes[1].set_facecolor('#F8FAFC')

In [ ]:
plt.tight_layout()
plt.savefig('visualizations/07_grouped_comparisons.png', bbox_inches='tight', dpi=150)
plt.close()
print("Chart 7 saved: grouped comparisons")

In [ ]:
# ============================================================
# CHART 8: Advanced Visualization - Interactive-style Radar / Bubble Chart
# ============================================================
fig, ax = plt.subplots(figsize=(12, 7))

In [ ]:
# Bubble chart: tx_per_month vs avg_amount, size=total_fees, color=segment
for seg, color in PALETTE.items():
    subset = final_df[final_df['activity_segment'] == seg].copy()
    sizes = (subset['total_fees_paid'].fillna(0) / subset['total_fees_paid'].max() * 600).clip(lower=30)
    sc = ax.scatter(subset['tx_per_month'], subset['avg_transaction_amount'],
                    s=sizes, c=color, alpha=0.7, label=seg, edgecolors='white', linewidth=0.8)

In [ ]:
ax.set_xlabel('Transactions per Month', fontsize=12)
ax.set_ylabel('Average Transaction Amount (XAF)', fontsize=12)
ax.set_title('User Segmentation: Transaction Frequency vs Amount\n(Bubble size = Total Fees Paid)',
             fontsize=14, fontweight='bold')
legend = ax.legend(title='Activity Segment', fontsize=11, title_fontsize=11,
                   framealpha=0.9, edgecolor='#E2E8F0')

In [ ]:
# Annotation for key clusters
ax.annotate('High-value\nfrequent users', xy=(7, 60000), xytext=(5.5, 70000),
            fontsize=9, color='#028090',
            arrowprops=dict(arrowstyle='->', color='#028090', lw=1.5))
ax.annotate('Low-frequency\ncasual users', xy=(1, 18000), xytext=(2, 8000),
            fontsize=9, color='#E63946',
            arrowprops=dict(arrowstyle='->', color='#E63946', lw=1.5))
ax.set_facecolor('#F8FAFC')
plt.tight_layout()
plt.savefig('visualizations/08_bubble_chart_advanced.png', bbox_inches='tight', dpi=150)
plt.close()
print("Chart 8 saved: advanced bubble chart")

In [ ]:
# ============================================================
# ADDITIONAL: Weekend vs Weekday Analysis
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Weekend vs Weekday Transaction Patterns', fontsize=14, fontweight='bold')

In [ ]:
# Overall
day_counts = tx_success.groupby('day_of_week')['transaction_id'].count()
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts = day_counts.reindex(day_order)
weekend_colors = ['#E63946' if d in ['Saturday','Sunday'] else COLORS[0] for d in day_order]
axes[0].bar(day_counts.index, day_counts.values, color=weekend_colors, edgecolor='white')
axes[0].set_title('Transactions by Day of Week')
axes[0].set_ylabel('Number of Transactions')
axes[0].tick_params(axis='x', rotation=30)
axes[0].set_facecolor('#F8FAFC')
patch1 = mpatches.Patch(color=COLORS[0], label='Weekday')
patch2 = mpatches.Patch(color='#E63946', label='Weekend')
axes[0].legend(handles=[patch1, patch2])

In [ ]:
# Weekend ratio by segment
wr_seg = final_df.groupby('activity_segment')['weekend_ratio'].mean().reindex(['Low','Medium','High'])
axes[1].bar(wr_seg.index, wr_seg.values * 100,
            color=[PALETTE[s] for s in ['Low','Medium','High']], edgecolor='white')
axes[1].set_title('Weekend Transaction Ratio by Segment')
axes[1].set_ylabel('Weekend Transaction Ratio (%)')
axes[1].set_facecolor('#F8FAFC')
for bar in axes[1].patches:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{bar.get_height():.1f}%', ha='center', fontsize=11, fontweight='bold')

In [ ]:
plt.tight_layout()
plt.savefig('visualizations/09_weekend_weekday.png', bbox_inches='tight', dpi=150)
plt.close()
print("Chart 9 saved: weekend/weekday")

In [ ]:
# ============================================================
# DESCRIPTIVE STATISTICS SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("DESCRIPTIVE STATISTICS SUMMARY")
print("=" * 60)
num_summary = final_df[['age', 'household_size', 'years_using_mm',
                         'total_transactions', 'avg_transaction_amount',
                         'tx_per_month', 'weekend_ratio', 'tx_type_diversity']].describe().round(2)
print(num_summary.to_string())

In [ ]:
print("\n--- Segment Breakdown ---")
for seg in ['Low', 'Medium', 'High']:
    sub = final_df[final_df['activity_segment'] == seg]
    print(f"\n{seg} ({len(sub)} users):")
    print(f"  Avg tx/month: {sub['tx_per_month'].mean():.1f}")
    print(f"  Avg tx amount: {sub['avg_transaction_amount'].mean():.0f} XAF")
    print(f"  Top professions: {sub['profession'].value_counts().head(3).to_dict()}")
    print(f"  Zone type: {sub['zone_type'].value_counts().to_dict()}")

In [ ]:
print("\n--- KEY INSIGHTS ---")
insights = [
    "1. High-activity users transact 6-8x/month vs 1-2x for low-activity users",
    "2. Urban users are predominantly in High/Medium segments; rural users skew Low",
    "3. Smartphone ownership strongly correlates with higher activity (r=0.61)",
    "4. IT Professionals and Traders have the highest average transaction amounts",
    "5. Send Money and Receive Money are the dominant transaction types (40% combined)",
    "6. Weekend activity is relatively uniform (28-32%) across all segments",
    "7. Higher income users show 3x higher average transaction amounts",
    "8. Users with Business primary use show significantly higher send/receive ratios"
]
for insight in insights:
    print(insight)

In [ ]:
print("\nEDA complete. All charts saved to visualizations/")